<!--
SPDX-FileCopyrightText:  PyPSA-Earth and PyPSA-Eur Authors
SPDX-License-Identifier: CC0-1.0
-->

# Analyse Kazakhstan results (Part 2)

Run this notebook from the **PyPSA-Earth project root** — the directory that contains `Snakefile` and `results/KZ/`. Paths such as `results/KZ/networks/...` are relative to that folder.

If you downloaded this file from the documentation, copy it to the project root before running. Do not run it from `doc/tutorials/use-cases/`.

In [ ]:
import pypsa
import pandas as pd
import matplotlib.pyplot as plt

# Load the network (path relative to project root)
n = pypsa.Network("results/KZ/networks/elec_s_10_ec_lcopt_6h.nc")
print(n)

### Annual demand

In [ ]:
# Compute total annual demand
weights = n.snapshot_weightings.generators

total_demand_TWh = n.loads_t.p_set.multiply(weights, axis=0).sum().sum() / 1e6
print(f"Total annual demand: {total_demand_TWh:.1f} TWh")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
n.loads_t.p_set.sum(axis=1).plot(ax=ax, title="Total load profile (MW)")
ax.set_ylabel("MW")
fig.tight_layout()

### Installed capacities

In [ ]:
# Get installed capacities
caps = n.statistics()["Installed Capacity"].dropna() / 1e3  # GW
print(caps.sort_values(ascending=False).to_string())

In [ ]:
stat = n.statistics()[["Installed Capacity", "Optimal Capacity"]].dropna() / 1e3  # GW
stat["New build"] = stat["Optimal Capacity"] - stat["Installed Capacity"]
print(stat.sort_values("Optimal Capacity", ascending=False).to_string())

In [ ]:
gen_caps = (
    n.generators.loc[n.generators.carrier != "load shedding"]  # exclude load shedding
    .groupby("carrier")["p_nom_opt"]
    .sum()
    .sort_values(ascending=False)
    / 1e3  # GW
)
print(gen_caps.to_string())
fig, ax = plt.subplots(figsize=(10, 4))
gen_caps.plot.bar(ax=ax, title="Optimal generation capacity by carrier (GW)")
ax.set_ylabel("GW")
fig.tight_layout()

### Annual generation by carrier

In [ ]:
# Estimate generation mix
gen_mwh = (
    n.generators_t.p.multiply(weights, axis=0).sum().groupby(n.generators.carrier).sum()
)

su_mwh = (
    n.storage_units_t.p.multiply(weights, axis=0)
    .sum()
    .groupby(n.storage_units.carrier)
    .sum()
)

gen = ((gen_mwh.add(su_mwh, fill_value=0)) / 1e6).sort_values(ascending=False)  # TWh
print(gen.to_string())
fig, ax = plt.subplots(figsize=(10, 4))
gen.plot.bar(ax=ax, title="Annual generation by carrier (TWh)")
ax.set_ylabel("TWh")
fig.tight_layout()

In [ ]:
supply = n.statistics()["Supply"].dropna() / 1e6  # TWh
print(supply.sort_values(ascending=False).to_string())

### Capacity factors

In [ ]:
gen_cf = (
    n.generators_t.p.divide(n.generators.p_nom_opt.clip(lower=1), axis=1)
    .mean()
    .groupby(n.generators.carrier)
    .mean()
)

su_cf = (
    n.storage_units_t.p.divide(n.storage_units.p_nom_opt.clip(lower=1), axis=1)
    .mean()
    .groupby(n.storage_units.carrier)
    .mean()
)

cf = gen_cf.add(su_cf, fill_value=0).sort_values(ascending=False)
print(cf.to_string())